# 15 — Neural Audio Codecs

## Objective

In this notebook, we will explore how neural audio codecs compress audio into discrete representations.

We will focus on the core idea behind modern codecs:

Audio Waveform
→ Encoder
→ Quantization
→ Discrete Audio Tokens
→ Decoder
→ Reconstructed Audio

We will also experiment with a simplified Residual Vector Quantization (RVQ) process and analyze the trade-off between compression and reconstruction quality.

## What we will explore

In this notebook, we will:

- generate a simple audio signal;
- simulate audio quantization;
- implement a simplified Residual Vector Quantization process;
- compare different numbers of codebooks;
- measure reconstruction error using Mean Squared Error;
- visualize the original and reconstructed signals;
- connect the experiment to codecs such as EnCodec, DAC, SNAC, and Mimi.

## Create a test audio signal

To study audio quantization, we will first generate a simple synthetic waveform.

The signal will combine several sine waves so that we have a clean and controlled audio example.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sample_rate = 16000
duration = 2.0

t = np.linspace(
    0,
    duration,
    int(sample_rate * duration),
    endpoint=False
)

signal = (
    0.6 * np.sin(2 * np.pi * 220 * t)
    + 0.3 * np.sin(2 * np.pi * 440 * t)
    + 0.1 * np.sin(2 * np.pi * 880 * t)
)

signal = signal / np.max(np.abs(signal))

print("Sample rate:", sample_rate)
print("Duration:", duration, "seconds")
print("Number of samples:", len(signal))

display(Audio(signal, rate=sample_rate))

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(t[:1600], signal[:1600])

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title("Original Audio Signal")

plt.show()

## Simple audio quantization

A neural audio codec compresses audio by mapping continuous values to a smaller set of discrete representations.

To understand this idea, we will first apply a simple scalar quantization to the waveform.

We will reduce the number of possible amplitude values and measure the reconstruction error.

In [ ]:
def quantize_signal(signal, levels=16):
    min_val = np.min(signal)
    max_val = np.max(signal)

    bins = np.linspace(min_val, max_val, levels)

    indices = np.digitize(signal, bins) - 1
    indices = np.clip(indices, 0, levels - 1)

    quantized = bins[indices]

    return quantized


quantized_signal = quantize_signal(signal, levels=16)

mse = np.mean((signal - quantized_signal) ** 2)

print("Quantization levels:", 16)
print("MSE:", round(mse, 6))

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(t[:1600], signal[:1600], label="Original")
plt.plot(t[:1600], quantized_signal[:1600], label="Quantized")

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title("Original vs Quantized Audio")

plt.legend()
plt.show()

### Observation

Using only 16 quantization levels, the waveform can still be reconstructed while using a much smaller set of discrete amplitude values.

The Mean Squared Error is approximately:

0.006058

This illustrates the basic compression trade-off:

fewer discrete values
→ stronger compression
→ higher reconstruction error

## Residual Vector Quantization

Residual Vector Quantization uses multiple quantization stages.

Each stage quantizes the residual error left by the previous stage.

The process can be represented as:

Original Signal
→ Quantizer 1
→ Residual
→ Quantizer 2
→ Residual
→ Quantizer 3
→ ...

The final reconstruction is obtained by adding the contribution of all quantization stages.

In [ ]:
def residual_quantization(signal, num_codebooks=4, levels=16):
    residual = signal.copy()
    reconstruction = np.zeros_like(signal)

    for _ in range(num_codebooks):
        quantized = quantize_signal(residual, levels=levels)

        reconstruction += quantized
        residual = signal - reconstruction

    return reconstruction


rvq_signal = residual_quantization(
    signal,
    num_codebooks=4,
    levels=16
)

rvq_mse = np.mean((signal - rvq_signal) ** 2)

print("RVQ MSE:", rvq_mse)
print("RVQ MSE scientific:", f"{rvq_mse:.12e}")

### Observation

Using 4 quantization stages, the reconstruction error becomes extremely small:

RVQ MSE ≈ 5.27 × 10⁻¹⁰

Each additional quantization stage encodes the residual error left by the previous stage.

As a result, the reconstruction becomes progressively more accurate.

In this simplified experiment, 4 stages are already enough to reconstruct the synthetic signal almost perfectly.

In [ ]:
codebook_counts = [1, 2, 4, 8]
mse_results = []

for n in codebook_counts:
    reconstructed = residual_quantization(
        signal,
        num_codebooks=n,
        levels=16
    )

    mse = np.mean((signal - reconstructed) ** 2)
    mse_results.append(mse)

    print(f"{n} codebook(s) -> MSE: {mse:.12e}")

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(codebook_counts, mse_results, marker="o")

plt.yscale("log")
plt.xlabel("Number of codebooks")
plt.ylabel("MSE (log scale)")
plt.title("RVQ Reconstruction Error")

plt.show()

### RVQ comparison

Increasing the number of codebooks dramatically reduces the reconstruction error.

Results:

- 1 codebook → MSE ≈ 6.06 × 10⁻³
- 2 codebooks → MSE ≈ 2.69 × 10⁻⁵
- 4 codebooks → MSE ≈ 5.27 × 10⁻¹⁰
- 8 codebooks → MSE ≈ 1.96 × 10⁻¹⁹

Each additional codebook encodes the residual error left by the previous stages.

As a result:

More codebooks
→ lower reconstruction error
→ better audio quality

However, using more codebooks also increases the amount of discrete information that must be stored or transmitted.

This illustrates the trade-off between bitrate and reconstruction quality in neural audio codecs.

## Compare reconstruction quality

We will compare the original waveform with reconstructions obtained using different numbers of codebooks.

This helps visualize how additional residual quantization stages improve the reconstructed signal.

In [ ]:
reconstruction_1 = residual_quantization(
    signal,
    num_codebooks=1,
    levels=16
)

reconstruction_4 = residual_quantization(
    signal,
    num_codebooks=4,
    levels=16
)

plt.figure(figsize=(12, 4))

plt.plot(t[:1600], signal[:1600], label="Original")
plt.plot(t[:1600], reconstruction_1[:1600], label="1 codebook")

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title("Original vs 1-Codebook Reconstruction")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))

plt.plot(t[:1600], signal[:1600], label="Original")
plt.plot(t[:1600], reconstruction_4[:1600], label="4 codebooks")

plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.title("Original vs 4-Codebook Reconstruction")
plt.legend()
plt.show()

## Modern Neural Audio Codecs

Several neural audio codecs use similar ideas but are optimized for different goals.

### EnCodec

EnCodec uses a neural encoder, Residual Vector Quantization, and a neural decoder.

Pipeline:

Audio
→ Encoder
→ RVQ Codebooks
→ Discrete Tokens
→ Decoder
→ Reconstructed Audio

It is designed for high-quality neural audio compression.

### DAC

DAC (Descript Audio Codec) is another neural audio codec focused on high-fidelity audio reconstruction.

It also uses learned representations and multiple quantization stages.

### SNAC

SNAC uses multiple temporal resolutions to represent audio efficiently.

This allows the codec to capture both coarse and fine acoustic information.

### Mimi

Mimi is designed for low-latency speech applications.

It is especially useful for real-time speech-to-speech systems and conversational AI.

### Key idea

Although their architectures differ, these codecs share the same general principle:

Audio
→ Neural Representation
→ Discrete Tokens
→ Reconstructed Audio

## Conclusion

In this notebook, we explored the core ideas behind neural audio codecs.

### What we learned

- audio signals can be converted into discrete representations;
- simple quantization introduces reconstruction error;
- Residual Vector Quantization progressively reduces this error;
- increasing the number of codebooks improves reconstruction quality;
- more codebooks also increase the amount of information that must be stored or transmitted;
- modern codecs such as EnCodec, DAC, SNAC, and Mimi use learned representations to compress audio efficiently.

### Key takeaway

Neural audio codecs transform continuous audio into compact discrete tokens.

This makes audio easier to compress, transmit, and process with modern generative AI systems such as speech models and multimodal Transformers.